# Phase 2.5 — Empirical timing calibration (n=3 stabilized greedy)

**Purpose:** measure the n=3 solver's **throughput (nodes/sec)** *and* **memory (MB/worker)** so we
can size the full sweep and pick budget tiers + how many parallel workers fit in RAM. Timing run,
not a correctness run.

**How to run on Colab**
1. Runtime → Change runtime type → **CPU** (High-RAM if offered).
2. Edit **only the CONFIG cell**. Set `REPO_URL`/`BRANCH`; pick `PROBE_WORKERS` (parallel probes),
   `PROBE_BUDGET`, sample sizes.
3. Runtime → **Run all**. Each probe result streams to `results/calibration.jsonl` **and is copied to
   Drive immediately** (so a stopped run still persists).
4. Read the **SUMMARY** table, download `calibration.jsonl` (or grab the Drive copy), and share both.

**Picking up new code after a push** (no runtime recreation): with `UPDATE_REPO=True` the setup cell
`git pull`s the branch every run, so just **Runtime → Restart runtime → Run all**. A *restart* gives a
fresh kernel (re-imports the updated `greedy_nrel`/`calibrate_probe`) while keeping the disk (no
re-clone, no re-`pip`), and the pull updates the files. You only need *Disconnect and delete runtime*
if the environment itself breaks. The setup cell prints the current commit hash — check it matches.

> Prereq: the branch must contain `greedy_nrel.py`, `stabilize.py`, `calibrate_probe.py`, and this
> notebook (committed & pushed).
> Memory note: each worker's `visited` grows with the budget (~7–8 KB/node), so **RAM, not cores, caps
> parallelism** at high budgets — the summary computes the ceiling for you.


In [ ]:
# ==================== PHASE 2.5 CALIBRATION — CONFIG (edit ONLY this cell) ====================
# Goal: measure n=3 throughput (nodes/sec) AND memory (MB/worker) to size the full sweep.

# --- repo / environment (Colab) ---
REPO_URL          = "https://github.com/Avi161/ACSolverX.git"
BRANCH            = "test/stable-ac-moves"
REPO_DIR          = "ACSolverX"
CLONE             = True               # clone the repo if REPO_DIR is absent
UPDATE_REPO       = True               # if REPO_DIR already exists, git-pull latest (reset --hard to
                                       #   origin/BRANCH) so a plain RESTART picks up new pushes WITHOUT
                                       #   deleting/recreating the runtime. Untracked files (your
                                       #   results/*.jsonl) are preserved. Set False to pin the current code.
MOUNT_DRIVE       = True
DRIVE_RESULTS_DIR = "/content/drive/MyDrive/acsolverx_calibration"

# --- what to probe ---
N_GEN                 = 3
ARMS                  = ["r1"]        # z=w arm(s). "r1" = representative; add "r2" for the heaviest arm.
MAX_LEN               = 24
USE_NULL_REVERT_BLOCK = True

# --- parallelism (fixes "box RAM barely used"): run probes across cores ---
PROBE_WORKERS = 4                      # parallel probes. Each uses ~PROBE_BUDGET*7.5KB of RAM, so keep
                                       #   PROBE_WORKERS * (that) <= box RAM. 1 = clean single-core baseline.

# --- budgets ---
PROBE_BUDGET      = 200_000            # nodes each HARD probe runs to. Throughput is ~stable with depth
                                       #   now, so 200k gives a solid nodes/sec fast. Raise toward your
                                       #   target tier to also measure memory at that tier.
CANDIDATE_BUDGETS = [100_000, 1_000_000]

# --- samples ---
HARD_DATASET = "ms_reps_unsolved"      # 261 hard reps -> data/ms_unsolved_reps/ms_reps_unsolved.txt
N_HARD       = 5
HARD_IDX     = None                    # None -> evenly spread; or explicit list e.g. [0, 65, 130, 195, 260]

EASY_DATASET = "1190MS"
N_EASY       = 5
EASY_IDX     = None
EASY_BUDGET  = 100_000

# --- projection: (label, total #presentations, approx #budget-exhausting "unsolved" cases) ---
PROJECT_OVER = [
    ("MS(1190)",          1190, 550),
    ("unsolved reps 261",  261, 261),
]
N_ARMS_TOTAL       = 4                 # x,y,r1,r2 -> whole-experiment compute multiplier
BOX_RAM_GB_OVERRIDE = None             # None -> auto-detect the box RAM; set a number to override
# =============================================================================================
print("config loaded.")


In [ ]:
# ==================== SETUP (clone / install / mount / import / detect box) ====================
import os, sys, subprocess, ast, time, json, resource, platform, statistics, multiprocessing as mp
def sh(cmd):
    print("$", cmd)
    p = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if p.stdout: print(p.stdout[-2000:])
    if p.returncode != 0 and p.stderr: print("STDERR:", p.stderr[-2000:])

try:
    import google.colab  # noqa
    IN_COLAB = True
except Exception:
    IN_COLAB = False
print("Colab:", IN_COLAB)

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        if CLONE:
            sh(f"git clone --branch {BRANCH} --depth 1 {REPO_URL} {REPO_DIR}")
    elif UPDATE_REPO:
        # pull latest without recloning; reset --hard leaves untracked results/*.jsonl intact
        sh(f"cd {REPO_DIR} && git fetch --depth 1 origin {BRANCH} && git reset --hard FETCH_HEAD")
    sh(f"cd {REPO_DIR} && git log -1 --oneline")   # <-- confirm which commit you're running
    sh("pip -q install numba numpy")
    if MOUNT_DRIVE:
        from google.colab import drive
        drive.mount("/content/drive")
        os.makedirs(DRIVE_RESULTS_DIR, exist_ok=True)
    REPO_ROOT = os.path.abspath(REPO_DIR)
else:
    REPO_ROOT = os.path.abspath(os.getcwd())
    while REPO_ROOT != "/" and not os.path.isdir(os.path.join(REPO_ROOT, "envs")):
        REPO_ROOT = os.path.dirname(REPO_ROOT)

ONEGEN = os.path.join(REPO_ROOT, "experiments", "stable_ac", "one_generator")
sys.path.insert(0, ONEGEN)
import greedy_nrel as gn
import stabilize as stab

RESULTS_DIR = os.path.join(ONEGEN, "results")
os.makedirs(RESULTS_DIR, exist_ok=True)
CALIB_PATH = os.path.join(RESULTS_DIR, "calibration.jsonl")

RAW = {
    "1190MS":           os.path.join(REPO_ROOT, "data", "1190MS.txt"),
    "ms_reps_unsolved": os.path.join(REPO_ROOT, "data", "ms_unsolved_reps", "ms_reps_unsolved.txt"),
}
def load_raw(name):
    with open(RAW[name]) as f:
        return [ast.literal_eval(l) for l in f if l.strip()]

CORES = os.cpu_count() or 1
def _box_ram_gb():
    try:
        with open("/proc/meminfo") as f:
            for line in f:
                if line.startswith("MemTotal"):
                    return int(line.split()[1]) / (1024.0 * 1024.0)  # KB -> GB
    except Exception:
        return None
BOX_RAM_GB = BOX_RAM_GB_OVERRIDE or _box_ram_gb()
print(f"repo : {REPO_ROOT}")
print(f"out  : {CALIB_PATH}")
print(f"box  : {CORES} cores, RAM = {BOX_RAM_GB and round(BOX_RAM_GB,1)} GB   (PROBE_WORKERS={PROBE_WORKERS})")


In [ ]:
# ==================== WARM UP numba (parent compiles; forked workers inherit the warm code) ====================
_warm = stab.stabilize_flat(load_raw("1190MS")[0], "r1")
t0 = time.time()
gn.solve_one(_warm, n_gen=N_GEN, max_len=MAX_LEN, max_nodes=2000)
print(f"warmup (numba JIT of all primitives) done in {time.time()-t0:.1f}s")


In [ ]:
# ==================== BUILD PROBE INDICES ====================
def spread_idx(n_total, k):
    if k <= 1: return [0]
    if k >= n_total: return list(range(n_total))
    return sorted({round(i * (n_total - 1) / (k - 1)) for i in range(k)})

hard_raw = load_raw(HARD_DATASET)
easy_raw = load_raw(EASY_DATASET)
hard_idx = HARD_IDX if HARD_IDX is not None else spread_idx(len(hard_raw), N_HARD)
easy_idx = EASY_IDX if EASY_IDX is not None else spread_idx(640, N_EASY)  # 1190MS solved region is idx<640
print(f"HARD ({HARD_DATASET}, {len(hard_raw)} lines) idx:", hard_idx)
print(f"EASY ({EASY_DATASET}) idx:", easy_idx)


In [ ]:
# ==================== RUN PROBES (parallel, resumable, incremental Drive copy) ====================
from calibrate_probe import probe as _probe_worker   # importable -> picklable under fork AND spawn

def _persist(rec):
    with open(CALIB_PATH, "a") as f:
        f.write(json.dumps(rec) + "\n"); f.flush(); os.fsync(f.fileno())
    if IN_COLAB and MOUNT_DRIVE:
        import shutil; shutil.copy(CALIB_PATH, os.path.join(DRIVE_RESULTS_DIR, "calibration.jsonl"))
    print(f"  [{rec['kind']:4}] {rec['dataset']:16} idx {rec['idx']:4} z={rec['arm']:2}: "
          f"solved={rec['solved']!s:5} nodes={rec['nodes_explored']:8} {rec['wall_time_s']:7.1f}s "
          f"{rec['nodes_per_sec']:6}/s  rss={rec['peak_rss_mb']:.0f}MB  reverts={rec['revert_hits']}")

# resumable: skip (kind,dataset,idx,arm,budget) already recorded
done = set()
if os.path.exists(CALIB_PATH):
    for l in open(CALIB_PATH):
        try:
            r = json.loads(l); done.add((r["kind"], r["dataset"], r["idx"], r["arm"], r["budget_nodes"]))
        except Exception:
            pass

tasks = []
for arm in ARMS:
    for idx in hard_idx:
        if ("hard", HARD_DATASET, idx, arm, PROBE_BUDGET) not in done:
            tasks.append(("hard", HARD_DATASET, idx, arm, PROBE_BUDGET, hard_raw[idx], N_GEN, MAX_LEN, USE_NULL_REVERT_BLOCK))
    for idx in easy_idx:
        if ("easy", EASY_DATASET, idx, arm, EASY_BUDGET) not in done:
            tasks.append(("easy", EASY_DATASET, idx, arm, EASY_BUDGET, easy_raw[idx], N_GEN, MAX_LEN, USE_NULL_REVERT_BLOCK))
print(f"{len(tasks)} probes to run on {PROBE_WORKERS} worker(s)  ({len(done)} already done)\n")

t0 = time.time()
if tasks and PROBE_WORKERS > 1:
    ctx = mp.get_context("fork")           # Colab/Linux: fresh child per task (clean per-probe RSS), warm numba
    with ctx.Pool(PROBE_WORKERS, maxtasksperchild=1) as pool:
        for rec in pool.imap_unordered(_probe_worker, tasks):
            _persist(rec)
else:
    for t in tasks:
        _persist(_probe_worker(t))
print(f"\ndone in {time.time()-t0:.0f}s wall.  results -> {CALIB_PATH}"
      + (f"  (+Drive: {DRIVE_RESULTS_DIR}/calibration.jsonl)" if (IN_COLAB and MOUNT_DRIVE) else ""))


In [ ]:
# ==================== SUMMARY & MEMORY-AWARE SWEEP PROJECTION ====================
recs = [json.loads(l) for l in open(CALIB_PATH) if l.strip()]
hard = [r for r in recs if r["kind"] == "hard"]
easy = [r for r in recs if r["kind"] == "easy"]
print(f"probes: {len(hard)} hard, {len(easy)} easy   (arms: {sorted(set(r['arm'] for r in recs))})\n")

exh     = [r["nodes_per_sec"] for r in hard if r["exhausted_budget"]]
allhard = [r["nodes_per_sec"] for r in hard]
nps_est = statistics.median(exh) if exh else (statistics.median(allhard) if allhard else 0)
kbpn    = statistics.median([r["peak_rss_mb"] * 1024.0 / r["nodes_explored"]
                             for r in hard if r["nodes_explored"] > 0]) if hard else 0
peak_mb = max((r["peak_rss_mb"] for r in hard), default=0)
ram_gb  = BOX_RAM_GB_OVERRIDE or BOX_RAM_GB or 12.0

print("THROUGHPUT (nodes/sec, per core):")
if allhard:
    print(f"  hard all      : median {statistics.median(allhard):.0f}  min {min(allhard):.0f}  max {max(allhard):.0f}")
print(f"  hard exhausting: median {nps_est:.0f}   <-- used for the projection ({len(exh)}/{len(hard)} exhausted budget)")
if not exh:
    print("  WARNING: no hard probe ran to full budget -> nodes/sec is a SOLVE-time number (optimistic).")
    print("           raise PROBE_BUDGET or pick harder HARD_IDX so probes exhaust the budget.")
if easy:
    print(f"  easy solved   : median wall {statistics.median([r['wall_time_s'] for r in easy]):.2f}s (small addend)")
print(f"\nMEMORY (per worker): peak at {PROBE_BUDGET:,} nodes = {peak_mb:.0f} MB ; rate ~{kbpn:.1f} KB/node ; box RAM {ram_gb:.0f} GB\n")

def usable_workers(tier):
    rss_gb = kbpn * tier / 1024.0 / 1024.0
    by_ram = int(ram_gb / rss_gb) if rss_gb > 0 else CORES
    return max(1, min(CORES, by_ram)), rss_gb, by_ram

if nps_est > 0:
    print("PROJECTED SWEEP TIME (per z=w arm; the fleet runs one arm per box in parallel):")
    hdr = f"  {'dataset':18} {'#unsolved':>9} {'tier':>11} {'GB/worker':>9} {'workers':>8} {'wall hrs/arm':>12}"
    print(hdr); print("  " + "-" * (len(hdr) - 2))
    for label, ntot, nuns in PROJECT_OVER:
        for tier in CANDIDATE_BUDGETS:
            w, rss_gb, by_ram = usable_workers(tier)
            wall_h = tier * nuns / nps_est / 3600.0 / w
            note = "" if by_ram >= CORES else f"  (RAM-capped: {by_ram} of {CORES} cores)"
            print(f"  {label:18} {nuns:9} {tier:11,} {rss_gb:9.1f} {w:8} {wall_h:12.1f}{note}")
    print()
    print(f"  Reading it: 'workers' = min(cores, RAM/GB-per-worker); 'wall hrs/arm' already divides by it.")
    print(f"  One arm per box in parallel -> FLEET wall ~= the largest 'wall hrs/arm'. Whole experiment")
    print(f"  COMPUTE = that x {N_ARMS_TOTAL} arms. If a tier is RAM-capped, that's the lever to watch")
    print(f"  (bigger box, or shard the idx range across boxes). Throughput is ~depth-stable, so a 200k")
    print(f"  probe projects the larger tiers reasonably; confirm memory by probing near the target tier.")
